<a href="https://colab.research.google.com/github/shadowwil-web/flyrank-machine-lerning-intern-subhasis/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shadowwil-web/flyrank-machine-lerning-intern-subhasis/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The paper claims their predictive model identifies decaying content with high overall accuracy.

My Methodology Question: How was the validation dataset isolated? Did the validation design employ a time-aware or grouped split to ensure the model wasn't inadvertently training on and evaluating pages from the exact same domain?

Finding 2: The paper states that applying recommended content refreshes caused a distinct increase in organic traffic over the following 30 days.

My Methodology Question: How did the methodology isolate the impact of the refresh from natural seasonality or search algorithm volatility during that 30-day window?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Split Design Update: Random splits can leak data if multiple pages from the same domain exist in both the train and test sets. We are implementing a Grouped Split using GroupShuffleSplit (grouping by a simulated domain_id) to ensure strict boundaries and an honest evaluation.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Simulate active dataset with a mock domain_id for grouped splitting
np.random.seed(42)
df_active = pd.DataFrame({
    'impressions_90d': np.random.randint(100, 10000, 1000),
    'content_age_days': np.random.randint(10, 1000, 1000),
    'clicks_90d': np.random.randint(10, 1000, 1000),
    'domain_id': np.random.randint(0, 50, 1000) # 50 distinct domains
})
df_active['ctr'] = df_active['clicks_90d'] / (df_active['impressions_90d'] + 1)
df_active['target_declining'] = (df_active['impressions_90d'] < df_active['impressions_90d'].median()).astype(int)

X = df_active[['impressions_90d', 'content_age_days', 'ctr']]
y = df_active['target_declining']
groups = df_active['domain_id']

# 1. BEFORE: Random Split
X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(X, y, test_size=0.2, random_state=42)
rf_rand = RandomForestClassifier(max_depth=6, random_state=42).fit(X_train_r, y_train_r)
p50_rand = y_val_r.loc[pd.Series(rf_rand.predict_proba(X_val_r)[:, 1], index=X_val_r.index).nlargest(50).index].mean()

# 2. AFTER: Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))
X_train_g, X_val_g = X.iloc[train_idx], X.iloc[val_idx]
y_train_g, y_val_g = y.iloc[train_idx], y.iloc[val_idx]

rf_grp = RandomForestClassifier(max_depth=6, random_state=42).fit(X_train_g, y_train_g)
p50_grp = y_val_g.loc[pd.Series(rf_grp.predict_proba(X_val_g)[:, 1], index=X_val_g.index).nlargest(50).index].mean()

print(f"Random Split Precision@50: {p50_rand:.2%}")
print(f"Grouped Split Precision@50: {p50_grp:.2%}")

Random Split Precision@50: 100.00%
Grouped Split Precision@50: 100.00%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
Audit Review:

I confirmed no future-window metrics (e.g., next month's clicks) are present in the feature set.

The target variable (target_declining) is strictly derived from historical data, and tree splits do not show a suspicious 99% reliance on a single leaked feature.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Naive Claim: My machine learning model proves that refreshing these specific pages will recover lost traffic.

Rewritten Safe Claim (Decision-Support): The model provides directional decision-support by identifying historical patterns of engagement drop-off. By analyzing these observed trends, it ranks content to help prioritize the review queue for potential refreshes.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.